# Call your own tools in the middle of a workflow

A support email comes in. To answer it, the model needs facts it cannot read from the email: the order's status from your database, and for a lost order, whether an agent approves a refund. Fields with `"source": "user"` are answered by your code, not the model. The call answers everything it can, stops with `status` `"needs_input"`, and `client.resume()` goes on with your answers.

1. Write the workflow: what the model answers, and what your side answers.
2. Answer from your side: a database lookup and an agent.
3. Run it on three emails.

Paste your TypeLLM API key in the cell below, then run all cells.

In [ ]:
%pip install -q "typellm>=0.7.0"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [ ]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

EMAILS = [
    "Hi, where is my order #8812? I ordered it last week and haven't heard anything. Thanks, Dana",
    "Order 8840 still hasn't arrived and it's been three weeks. Tracking hasn't moved since it left the "
    "warehouse. I'd like my money back. - Sam",
    "How do I change the email address on my account? - Lee",
]

## Write the workflow

The model reads the order number. Your database gives the order's status, only when there is an order number. An agent decides a refund, only for a lost order. The model then writes the reply that fits: about the order, about the refund, or a general one when there is no order.

In [ ]:
QUESTIONS = {
    "order_id": {"type": ["string", "null"], "instructions": "The order number in the email, digits only."},
    "order_status": {
        "type": "string",
        "enum": ["processing", "shipped", "delivered", "lost", "not_found"],
        "source": "user",
        "instructions": "The order's status in our system.",
        "when": {"order_id": {"ne": None}},
    },
    "refund_approved": {
        "type": "boolean",
        "source": "user",
        "instructions": "Has a support agent approved a refund?",
        "when": {"order_status": "lost"},
    },
    "reply": {
        "type": "string",
        "instructions": "Reply to the customer about their order's status.",
        "when": {"order_status": {"not_in": ["lost"]}},
    },
    "refund_reply": {
        "type": "string",
        "instructions": "Reply to the customer about their lost order and the refund decision.",
        "depends_on": ["refund_approved"],
    },
    "general_reply": {
        "type": "string",
        "instructions": "Reply to the customer.",
        "when": {"order_id": None},
    },
}

## Answer from your side

Each field with `"source": "user"` gets a function that answers it from what the call has so far. Here a dictionary stands in for the order database, and the agent approves every refund; in your code, query the database, call your API, or put the question in front of a person.

In [ ]:
ORDERS = {"8812": "shipped", "8840": "lost"}  # your order database


def lookup_order(result):
    return ORDERS.get(result["order_id"], "not_found")


def ask_agent(result):
    print(f"  agent: refund order {result['order_id']}, which is {result['order_status']}? yes")
    return True


ANSWER = {"order_status": lookup_order, "refund_approved": ask_agent}


def run(email):
    """Answer an email: stop for what our side answers, answer it, and go on until done."""
    response = client.generate(context=email, questions=QUESTIONS)
    while response.status == "needs_input":
        print(f"  stopped for {list(response.needs_input)}, with {response.result}")
        answers = {name: ANSWER[name](response.result) for name in response.needs_input}
        response = client.resume(response, answers)
    return response

## Run it on three emails

The first stops once, for the order's status. The second stops twice: for the status, then, since the order is lost, for the agent. The third has no order number, so it asks nothing of our side and finishes in one call.

In [ ]:
for email in EMAILS:
    print(email)
    response = run(email)
    for name, answer in response.result.items():
        print(f"  {name}: {answer}")
    print()

Hi, where is my order #8812? I ordered it last week and haven't heard anything. Thanks, Dana
  stopped for ['order_status'], with {'order_id': '8812'}
  order_id: 8812
  order_status: shipped
  reply: Hi Dana, your order #8812 has been shipped. You should receive it soon. Thanks for your patience!

Order 8840 still hasn't arrived and it's been three weeks. Tracking hasn't moved since it left the warehouse. I'd like my money back. - Sam
  stopped for ['order_status'], with {'order_id': '8840'}
  stopped for ['refund_approved'], with {'order_id': '8840', 'order_status': 'lost'}
  agent: refund order 8840, which is lost? yes
  order_id: 8840
  order_status: lost
  refund_approved: True
  refund_reply: Hi Sam, I am very sorry to hear that your order (8840) has not arrived. I have checked our system and confirmed that the package is marked as lost. Since the item is no longer in transit, I have approved a full refund for you. You should see the funds returned to your original payment method